In [ ]:
import requests
import os

DATA_URL = "https://people.sc.fsu.edu/~jburkardt/data/csv/hw_200.csv"
RAW_PATH = "/home/jovyan/sales-data-pipeline/data/raw/sales_raw.csv"

os.makedirs("/home/jovyan/sales-data-pipeline/data/raw", exist_ok=True)
print("Ingesting data from source...")
response = requests.get(DATA_URL)
with open(RAW_PATH, "wb") as f:
    f.write(response.content)
print("✅ Data saved to data/raw/sales_raw.csv")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round, when

spark = SparkSession.builder \
    .appName("SalesPipeline") \
    .getOrCreate()

df = spark.read.csv(
    "/home/jovyan/sales-data-pipeline/data/raw/sales_raw.csv",
    header=True, inferSchema=True)

df_clean = df \
    .withColumnRenamed('Index', 'id') \
    .withColumnRenamed(' Height(Inches)"', 'height_inches') \
    .withColumnRenamed(' "Weight(Pounds)"', 'weight_pounds')

df_transformed = df_clean \
    .withColumn("height_cm", round(col("height_inches") * 2.54, 2)) \
    .withColumn("weight_kg", round(col("weight_pounds") * 0.453592, 2)) \
    .withColumn("bmi", round(
        col("weight_kg") / ((col("height_cm") / 100) ** 2), 2)) \
    .withColumn("bmi_category",
        when(col("bmi") < 18.5, "Underweight")
        .when(col("bmi") < 25.0, "Normal")
        .otherwise("Overweight"))

df_transformed.show(5)
print("✅ Transformation complete!")

In [ ]:
import pandas as pd

df_transformed.write.csv(
    "/home/jovyan/sales-data-pipeline/data/processed/",
    header=True, mode="overwrite")

df = pd.read_csv(
    "/home/jovyan/sales-data-pipeline/data/processed/part-00000-b2dbe100-2ed8-4cd7-b870-8ec6efe0e961-c000.csv")

df.to_csv(
    "/home/jovyan/sales-data-pipeline/data/processed/final_output.csv",
    index=False)

print("✅ Final data saved!")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

In [ ]:
print("Sample output:")
df.head()